In [ ]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
pip('insightface')
pip('onnxruntime-gpu')
import insightface
app = insightface.app.FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0)
print('✓ InsightFace buffalo_l загружен')

In [ ]:
import torch, torch.nn.functional as F, numpy as np, os, cv2, glob
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from ModelNew_PyTorchFF import FaceSwapModel
torch.cuda.empty_cache()
model = FaceSwapModel(
    img_size=256, id_lambda=8.0, perceptual_lambda=1.0,
    self_recon_lambda=5.0, l1_lambda=0.5, pose_lambda=1.0)
model.build_gan()
print()
print("="*60)
print("ТЕСТ 1: Style injection (AdaIN работает?)")
print("="*60)
model.generator.eval()
with torch.no_grad():
    B  = torch.randn(1, 3, 256, 256).to(DEVICE)
    A1 = torch.randn(1, 3, 256, 256).to(DEVICE)
    A2 = torch.randn(1, 3, 256, 256).to(DEVICE)
    out1 = model.generator(B, A1)
    out2 = model.generator(B, A2)
    out3 = model.generator(B, A1)
    diff_styles = (out1 - out2).abs().mean().item()
    diff_same   = (out1 - out3).abs().mean().item()
print(f"  G(B,A1) vs G(B,A2) разница: {diff_styles:.6f}  (должна быть > 0.01)")
print(f"  G(B,A1) vs G(B,A1) разница: {diff_same:.8f}   (должна быть ~0)")
if diff_styles > 0.01:
    print("  ✓ AdaIN РАБОТАЕТ — стиль влияет на выход")
else:
    print("  ✗ AdaIN СЛОМАН — стиль игнорируется (причина коллапса!)")
print()
print("="*60)
print("ТЕСТ 2: Градиент через id_loss (perc fake vs A)")
print("="*60)
model.generator.train()
B  = torch.randn(2, 3, 256, 256).to(DEVICE)
A  = torch.randn(2, 3, 256, 256).to(DEVICE)
fake = model.generator(B, A)
id_loss = model.perc_loss(fake, A)
id_loss.backward()
eid_grads = [p.grad for p in model.generator.eid.parameters()
             if p.grad is not None]
adain_grads = []
for res in model.generator.res_blocks:
    for p in res.adain1.style.parameters():
        if p.grad is not None:
            adain_grads.append(p.grad.abs().mean().item())
print(f"  eid градиентов: {len(eid_grads)}")
print(f"  AdaIN style градиент (среднее): "
      f"{np.mean(adain_grads):.6f}" if adain_grads else "  AdaIN: НЕТ ГРАДИЕНТОВ")
if adain_grads and np.mean(adain_grads) > 1e-8:
    print("  ✓ Градиент проходит через AdaIN — id_loss обучает стиль")
else:
    print("  ✗ Градиент НЕ проходит через AdaIN — id_loss бесполезен!")
model.generator.zero_grad()
print()
print("="*60)
print("ТЕСТ 3: Collapse check — G(B,A) vs B на реальных фото")
print("="*60)
PHOTOS_DIR = "extracted_photos_ff"
FF_VIDEOS  = os.path.join("original_sequences","youtube","c23","videos")
photos = sorted(glob.glob(os.path.join(PHOTOS_DIR, "*.jpg")))[:4]
videos = sorted(glob.glob(os.path.join(FF_VIDEOS, "*.mp4")))[:4]
def read_face(path_or_frame, size=256):
    if isinstance(path_or_frame, str):
        img = cv2.imread(path_or_frame)
    else:
        img = path_or_frame
    if img is None: return None
    img = cv2.cvtColor(cv2.resize(img,(size,size)), cv2.COLOR_BGR2RGB)
    return (torch.from_numpy(img.astype(np.float32)/127.5-1.0)
            .permute(2,0,1).unsqueeze(0).to(DEVICE))
model.generator.eval()
collapse_scores = []
arc_sims = []
if photos and videos:
    for i, (ph, vi) in enumerate(zip(photos[:4], videos[:4])):
        tA = read_face(ph)
        cap = cv2.VideoCapture(vi)
        cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
        ret, frame = cap.read(); cap.release()
        if not ret or tA is None: continue
        tB = read_face(frame)
        with torch.no_grad():
            fake = model.generator(tB, tA)
            cs = (fake - tB).abs().mean().item()
            collapse_scores.append(cs)
            ea = model.id_encoder(tA)
            ef = model.id_encoder(fake)
            sim = F.cosine_similarity(ea, ef).mean().item()
            arc_sims.append(sim)
            print(f"  Пара {i+1}: collapse_score={cs:.4f}  ArcSim(fake,A)={sim:.3f}")
    avg_cs  = np.mean(collapse_scores) if collapse_scores else 0
    avg_sim = np.mean(arc_sims) if arc_sims else 0
    print(f"  Среднее collapse_score: {avg_cs:.4f}")
    print(f"  Среднее ArcSim(fake,A): {avg_sim:.3f}")
    if avg_cs < 0.02:
        print("  ✗ КОЛЛАПС: fake почти идентичен B")
    elif avg_cs < 0.05:
        print("  ⚠ Слабый своп: небольшие изменения")
    else:
        print("  ✓ Своп активен: значимые изменения")
    if avg_sim > 0.5:
        print("  ✓ Identity: ArcFace видит сходство с A")
    elif avg_sim > 0.2:
        print("  ⚠ Identity: слабое сходство с A")
    else:
        print("  ✗ Identity: fake не похож на A")
else:
    print("  ⚠ Нет данных для теста — проверь пути PHOTOS_DIR и FF_VIDEOS")
print()
print("="*60)
print("ТЕСТ 4: Баланс лоссов на одном батче")
print("="*60)
if photos and videos:
    tA2_list, tB2_list = [], []
    for ph, vi in zip(photos[:2], videos[:2]):
        tA2 = read_face(ph)
        cap = cv2.VideoCapture(vi)
        cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
        ret, frame = cap.read(); cap.release()
        tB2 = read_face(frame)
        if tA2 is not None and tB2 is not None:
            tA2_list.append(tA2); tB2_list.append(tB2)
    if tA2_list:
        bA = torch.cat(tA2_list)
        bB = torch.cat(tB2_list)
        model.generator.train()
        ep0_b,ep1_b,ep2_b,ep3_b,ep4_b,ep5_b = model.generator.encode_pose(bB)
        id_feat = model.generator.eid(bA)
        style_a = model.generator.eid_proj(id_feat)
        fake_cross = model.generator.decode(ep0_b,ep1_b,ep2_b,ep3_b,ep4_b,ep5_b,style_a)
        id_loss   = model.perc_loss(fake_cross, bA)
        perc_loss = model.perc_loss(fake_cross, bB)
        ep0_a,ep1_a,ep2_a,ep3_a,ep4_a,ep5_a = model.generator.encode_pose(bA)
        style_a2 = model.generator.eid_proj(model.generator.eid(bA))
        fake_self = model.generator.decode(ep0_a,ep1_a,ep2_a,ep3_a,ep4_a,ep5_a,style_a2)
        sr_loss = F.l1_loss(fake_self, bA)
        ep0_f,ep1_f,ep2_f,ep3_f,ep4_f,ep5_f = model.generator.encode_pose(fake_cross)
        pose_loss = F.l1_loss(ep5_f, ep5_b.detach())
        anticollapse = F.relu(0.05 - perc_loss)
        print(f"  id_loss   (perc fake vs A): {id_loss.item():.4f}  * {model.id_lambda} = {id_loss.item()*model.id_lambda:.4f}")
        print(f"  perc_loss (perc fake vs B): {perc_loss.item():.4f}  * {model.perceptual_lambda} = {perc_loss.item()*model.perceptual_lambda:.4f}")
        print(f"  sr_loss   (G(A,A) vs A):   {sr_loss.item():.4f}  * {model.self_recon_lambda} = {sr_loss.item()*model.self_recon_lambda:.4f}")
        print(f"  pose_loss (ep5 feat):       {pose_loss.item():.4f}  * {model.pose_lambda} = {pose_loss.item()*model.pose_lambda:.4f}")
        print(f"  anticollapse:               {anticollapse.item():.4f}  * 5.0 = {anticollapse.item()*5.0:.4f}")
        total = (id_loss*model.id_lambda + perc_loss*model.perceptual_lambda +
                 sr_loss*model.self_recon_lambda + pose_loss*model.pose_lambda +
                 anticollapse*5.0)
        print(f"  ─────────────────────────────────────────────────")
        print(f"  g_total: {total.item():.4f}")
        id_pct = id_loss.item()*model.id_lambda / total.item() * 100
        print(f"  Доля id_loss в g_total: {id_pct:.1f}%  (хорошо если > 40%)")
        if id_pct > 40:
            print("  ✓ Identity доминирует в loss — своп будет работать")
        elif id_pct > 20:
            print("  ⚠ Identity присутствует, но слабо")
        else:
            print("  ✗ Identity слишком мало — модель будет копировать B")
print()
print("="*60)
print("ИТОГ ДИАГНОСТИКИ")
print("="*60)
print("Если все тесты ✓ — запускай обучение.")
print("Если есть ✗ — не запускай, сначала исправь.")
print()
print("Ключевые пороги для хорошего обучения:")
print("  Тест 1 (AdaIN):    diff_styles > 0.01")
print("  Тест 2 (Gradient): adain_grad  > 1e-8")
print("  Тест 3 (Collapse): collapse_score > 0.05")
print("  Тест 4 (Balance):  id_loss доля > 40%")


In [ ]:
import ModelNew_PyTorchFF
print(ModelNew_PyTorchFF.__file__)

In [ ]:
import insightface
app = insightface.app.FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0)
print('OK')
import os
print(os.path.abspath('ModelNew_PyTorchFF++.py'))

In [ ]:
import os
path = os.path.join('output', 'trained_models', 'generator_best.pth')
if os.path.exists(path):
    os.remove(path)
    print('✓ Старые веса удалены')

In [ ]:
import os
FF_VIDEOS_DIR    = os.path.join('original_sequences','youtube','c23','videos')
NUM_PAIRS        = 100
MAX_FRAMES       = 100
BATCH_SIZE       = 8
PHOTOS_DIR       = 'extracted_photos_ff'
PHASE2_EPOCHS    = 30
PHASE3_EPOCHS    = 40
D_PRETRAIN_STEPS = 200
SOURCE_PHOTO     = os.path.join(PHOTOS_DIR, 'person_000.jpg')
OUTPUT_VIDEO     = os.path.join('output', 'deepfake_result.mp4')
for d in [PHOTOS_DIR,
          os.path.join('output','trained_models'),
          os.path.join('output','progress')]:
    os.makedirs(d, exist_ok=True)
print('✓ Config OK')

In [ ]:
import os, glob, cv2, torch, urllib.request, bz2
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
print('[1/8] GPU...')
assert torch.cuda.is_available(), 'GPU не найден!'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU : {torch.cuda.get_device_name(0)}')
print(f'  VRAM: {vram:.1f} GB')
if vram < 6:
    BATCH_SIZE = 4
print('[2/8] Dlib predictor...')
PREDICTOR = 'shape_predictor_68_face_landmarks.dat'
if not os.path.exists(PREDICTOR):
    url = 'http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2'
    urllib.request.urlretrieve(url, PREDICTOR + '.bz2')
    with bz2.BZ2File(PREDICTOR + '.bz2','rb') as fi, open(PREDICTOR,'wb') as fo:
        fo.write(fi.read())
    os.remove(PREDICTOR + '.bz2')
print(f'  {os.path.getsize(PREDICTOR)/1e6:.1f} MB')
print('[3/8] Подготовка данных...')
all_videos = sorted(glob.glob(os.path.join(FF_VIDEOS_DIR, '*.mp4')))
assert len(all_videos) >= NUM_PAIRS * 2, f'Нужно {NUM_PAIRS*2} видео, найдено {len(all_videos)}'
videos_A = all_videos[:NUM_PAIRS]
videos_B = all_videos[NUM_PAIRS:NUM_PAIRS*2]
photo_paths, video_paths = [], []
for i, (va, vb) in enumerate(zip(videos_A, videos_B)):
    pp = os.path.join(PHOTOS_DIR, f'person_{i:03d}.jpg')
    if not os.path.exists(pp):
        cap = cv2.VideoCapture(va)
        best_frame, best_lap = None, 0
        for _ in range(60):
            ret, frame = cap.read()
            if not ret: break
            lap = cv2.Laplacian(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            if lap > best_lap:
                best_lap, best_frame = lap, frame.copy()
        cap.release()
        if best_frame is not None:
            cv2.imwrite(pp, best_frame)
    if os.path.exists(pp):
        photo_paths.append(pp)
        video_paths.append(vb)
    if (i+1) % 25 == 0:
        print(f'  {i+1}/{NUM_PAIRS} пар...')
print(f'  {len(photo_paths)} пар готово')
print('[4/8] Модель v13...')
from ModelNew_PyTorchFF import FaceSwapModel
torch.cuda.empty_cache()
model = FaceSwapModel(
    img_size=256, id_lambda=10.0, perceptual_lambda=1.0,
    self_recon_lambda=5.0, l1_lambda=0.5, pose_lambda=1.0)
model.build_gan()
model.g_scaler = torch.amp.GradScaler(init_scale=256.0)
phase1 = os.path.join('output','trained_models','generator_best.pth')
if os.path.exists(phase1):
    model.load_model(phase1)
    print('  Загружены предыдущие веса')
else:
    print('  Обучение с нуля')
print('[5/8] Загрузка датасета...')
model.load_paired_data(photo_paths, video_paths, max_frames_per_video=MAX_FRAMES)
def pretrain_d(model, steps, batch_size=16, target_loss=0.15, max_retries=5):
    total_steps, last_loss = 0, 0.0
    for attempt in range(max_retries):
        run_steps = steps if attempt == 0 else 100
        print(f'  D pretrain: {run_steps} шагов (попытка {attempt+1})...')
        model.generator.eval()
        dl = model.create_dataloader(batch_size=batch_size, shuffle=True, augment=False)
        it = iter(dl)
        for _ in range(run_steps):
            try:
                bB, bA, _ = next(it)
            except StopIteration:
                it = iter(dl); bB, bA, _ = next(it)
            bB, bA = bB.to(model.device), bA.to(model.device)
            model.disc_optimizer.zero_grad()
            with torch.no_grad():
                fake = model.generator(bB, bA)
            dr1, dr2 = model.discriminator(bB, bA)
            df1, df2 = model.discriminator(bB, fake)
            d_loss = (
                (F.relu(1.0-dr1).mean() + F.relu(1.0+df1).mean())*0.5 +
                (F.relu(1.0-dr2).mean() + F.relu(1.0+df2).mean())*0.5
            ) * 0.5
            d_loss.backward()
            model.disc_optimizer.step()
            last_loss = d_loss.item()
            total_steps += 1
        model.generator.train()
        print(f'    D loss: {last_loss:.4f}')
        if last_loss >= target_loss:
            break
    print(f'  Итого шагов: {total_steps}, D loss: {last_loss:.4f}')
    return last_loss
print('[6/8] ФАЗА 2: Слабый GAN...')
pretrain_d(model, steps=D_PRETRAIN_STEPS)
model.train(epochs=PHASE2_EPOCHS, batch_size=BATCH_SIZE, adv_lambda=0.1, save_interval=5)
print('[7/8] ФАЗА 3: Полный GAN...')
pretrain_d(model, steps=D_PRETRAIN_STEPS)
model.train(epochs=PHASE3_EPOCHS, batch_size=BATCH_SIZE, adv_lambda=0.5, save_interval=5)
print('[8/8] Сохранение...')
model.save_model()
model.plot_training_history()
best = os.path.join('output','trained_models','generator_best.pth')
if os.path.exists(best):
    model.load_model(best)
model.swap_face_in_video_PATCHED(
    source_photo_path=SOURCE_PHOTO,
    target_video_path=videos_B[0],
    output_path=OUTPUT_VIDEO)
cap = cv2.VideoCapture(OUTPUT_VIDEO)
ret, frame = cap.read(); cap.release()
if ret:
    fig, axes = plt.subplots(1, 2, figsize=(14,6))
    src = cv2.imread(SOURCE_PHOTO)
    axes[0].imshow(cv2.cvtColor(src,   cv2.COLOR_BGR2RGB)); axes[0].set_title('Источник (A)'); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); axes[1].set_title('Deepfake');     axes[1].axis('off')
    plt.suptitle('FaceSwapGAN v13'); plt.tight_layout()
    plt.savefig(os.path.join('output','deepfake_first_frame.jpg'), dpi=150)
    plt.show()
print('ВСЁ ГОТОВО! Deepfake:', OUTPUT_VIDEO)

In [ ]:
import os, glob, numpy as np, torch, torch.nn.functional as F, cv2, json
from torchvision import models
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim_fn
from scipy import linalg
WEIGHTS_PATH   = os.path.join('output','trained_models','generator_best.pth')
TEST_PHOTOS    = 'extracted_photos_ff'
TEST_VIDEOS    = os.path.join('original_sequences','youtube','c23','videos')
NUM_TEST_PAIRS = 20
FRAMES_PER_VID = 10
MAX_SCAN       = 300
MIN_FACE_SIZE  = 60
MIN_SHARPNESS  = 80.0
IMG_SIZE       = 256
BATCH_FID      = 32
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from ModelNew_PyTorch import FaceSwapModel
ev = FaceSwapModel(img_size=IMG_SIZE)
ev.build_gan()
ev.load_model(WEIGHTS_PATH)
ev.generator.eval()
id_enc = ev.id_encoder
perc_fn = ev.perc_loss
det = ev.face_detector
print('Модель загружена')
all_videos = sorted(glob.glob(os.path.join(TEST_VIDEOS,'*.mp4')))
all_photos = sorted(glob.glob(os.path.join(TEST_PHOTOS,'*.jpg')))
NUM_PAIRS  = 100
videos_B_all = all_videos[NUM_PAIRS:NUM_PAIRS*2]
tp = all_photos[-NUM_TEST_PAIRS:] if len(all_photos)>=NUM_TEST_PAIRS else all_photos
tv = videos_B_all[-NUM_TEST_PAIRS:] if len(videos_B_all)>=NUM_TEST_PAIRS else videos_B_all
n  = min(len(tp), len(tv), NUM_TEST_PAIRS)
print(f'Тестовых пар: {n}')
for i in range(min(3, n)):
    print(f'  {os.path.basename(tp[i])} ↔ {os.path.basename(tv[i])}')
def to_t(bgr, size=IMG_SIZE):
    rgb = cv2.cvtColor(cv2.resize(bgr,(size,size)), cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb.astype(np.float32)/127.5-1.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
def face_crop(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    r   = det.detect_face(rgb)
    h,w = bgr.shape[:2]
    fw,fh = r.right()-r.left(), r.bottom()-r.top()
    if fw>w*0.95 and fh>h*0.95: return None, 0
    if fw<MIN_FACE_SIZE or fh<MIN_FACE_SIZE: return None, 0
    pad = int(max(fw,fh)*0.2)
    x1,y1 = max(0,r.left()-pad), max(0,r.top()-pad)
    x2,y2 = min(w,r.right()+pad), min(h,r.bottom()+pad)
    crop = bgr[y1:y2,x1:x2]
    if crop.shape[0]<4 or crop.shape[1]<4: return None, 0
    sh = cv2.Laplacian(cv2.cvtColor(crop,cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
    if sh < MIN_SHARPNESS: return None, 0
    return crop, sh*(fw*fh)
def good_frames(vpath, n_frames):
    cap = cv2.VideoCapture(vpath)
    tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sc  = min(MAX_SCAN, tot)
    step = max(1, tot//sc)
    cands = []
    for i in range(sc):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i*step)
        ret, f = cap.read()
        if not ret: break
        c, s = face_crop(f)
        if c is not None: cands.append((s,c))
    cap.release()
    cands.sort(key=lambda x: x[0], reverse=True)
    return [c for _,c in cands[:n_frames]]
rB, fk, rA = [], [], []
with torch.no_grad():
    for i in tqdm(range(n), desc='swap'):
        img = cv2.imread(tp[i])
        if img is None: continue
        ca, _ = face_crop(img)
        tA = to_t(ca if ca is not None else img)
        crops = good_frames(tv[i], FRAMES_PER_VID)
        if not crops: continue
        for crop in crops:
            tB = to_t(crop)
            f  = ev.generator(tB, tA)
            rB.append(tB.cpu()); fk.append(f.cpu()); rA.append(tA.cpu())
print(f'Изображений: {len(fk)}')
def id_loss(fakes, reals, bs=8):
    vals = []
    for i in range(0,len(fakes),bs):
        bf = torch.cat(fakes[i:i+bs]).to(DEVICE)
        ba = torch.cat(reals[i:i+bs]).to(DEVICE)
        with torch.no_grad():
            vals.append((1-F.cosine_similarity(id_enc(bf),id_enc(ba))).mean().item())
    return float(np.mean(vals))
id_v = id_loss(fk, rA)
print(f'Identity Loss  : {id_v:.4f}')
def perc_loss(fakes, reals, bs=8):
    vals = []
    for i in range(0,len(fakes),bs):
        bf = torch.cat(fakes[i:i+bs]).to(DEVICE)
        bb = torch.cat(reals[i:i+bs]).to(DEVICE)
        with torch.no_grad():
            vals.append(perc_fn(bf,bb).item())
    return float(np.mean(vals))
perc_v = perc_loss(fk, rB)
print(f'Perceptual Loss: {perc_v:.4f}')
def tgray(t):
    arr = ((t[0].permute(1,2,0).numpy()+1)*127.5).clip(0,255).astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
ssim_v = float(np.mean([ssim_fn(tgray(f),tgray(b),data_range=255) for f,b in zip(fk,rB)]))
print(f'SSIM           : {ssim_v:.4f}')
class IncFeat(torch.nn.Module):
    def __init__(self):
        super().__init__()
        inc = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT, transform_input=False)
        self.net = torch.nn.Sequential(
            inc.Conv2d_1a_3x3,inc.Conv2d_2a_3x3,inc.Conv2d_2b_3x3,torch.nn.MaxPool2d(3,2),
            inc.Conv2d_3b_1x1,inc.Conv2d_4a_3x3,torch.nn.MaxPool2d(3,2),
            inc.Mixed_5b,inc.Mixed_5c,inc.Mixed_5d,
            inc.Mixed_6a,inc.Mixed_6b,inc.Mixed_6c,inc.Mixed_6d,inc.Mixed_6e,
            inc.Mixed_7a,inc.Mixed_7b,inc.Mixed_7c,
            torch.nn.AdaptiveAvgPool2d((1,1)),torch.nn.Flatten())
    def forward(self,x):
        return self.net(F.interpolate((x+1)/2,(299,299),mode="bilinear",align_corners=False))
def get_feats(tl, m, bs=BATCH_FID):
    m.eval(); out=[]
    for i in range(0,len(tl),bs):
        b = torch.cat(tl[i:i+bs]).to(DEVICE)
        with torch.no_grad(): out.append(m(b).cpu().numpy())
    return np.concatenate(out,0)
inc = IncFeat().to(DEVICE)
rf = get_feats(rB, inc); ff = get_feats(fk, inc)
mr,mf = rf.mean(0), ff.mean(0)
sr,sf = np.cov(rf,rowvar=False), np.cov(ff,rowvar=False)
cm = linalg.sqrtm(sr@sf)
if np.iscomplexobj(cm): cm = cm.real
fid_v = float((mr-mf)@(mr-mf) + np.trace(sr+sf-2*cm))
print(f'FID            : {fid_v:.2f}')
print("="*50)
print("  ФИНАЛЬНЫЕ МЕТРИКИ")
print("="*50)
print(f"  Identity Loss   {id_v:>8.4f}   (< 0.30)")
print(f"  Perceptual Loss {perc_v:>8.4f}   (< 1.00)")
print(f"  SSIM            {ssim_v:>8.4f}   (> 0.70)")
print(f"  FID             {fid_v:>8.2f}   (< 50)")
print("="*50)
with open(os.path.join('output','test_metrics.json'),'w',encoding='utf-8') as f:
    json.dump({'weights':WEIGHTS_PATH,'n_images':len(fk),
               'identity_loss':round(id_v,4),'perceptual_loss':round(perc_v,4),
               'ssim':round(ssim_v,4),'fid':round(fid_v,2)},f,indent=2,ensure_ascii=False)
print('Метрики сохранены: output/test_metrics.json')